In [7]:
import numpy as np

In [8]:
class RandomWalk:
    """19-state random walk. Fixed uniform-random policy => PREDICTION, like ex 3.
    step() ignores its action, and true_V() solves the MDP exactly (no sampling)."""

    N = 19

    def __init__(self, rng=None):
        self.rng = rng
        self.s = None
        self.nA = 2
        self.P = self._build_model()

    def reset(self):
        self.s = (self.N + 1) // 2                      # start in the middle
        return self.s

    def _move(self, s: int, a: int):
        s += [-1, 1][a]
        return s   

    def _build_model(self) -> dict:
        P = {s: {a: [] for a in range(self.nA)} for s in range(self.N)}
        for s in range(self.N):
            for a in range(self.nA):
                outcomes = {0: 0.5, 1: 0.5}
                agg: dict[int, list] = {}    # next_state -> [prob, reward, done]
                for act, prob in outcomes.items():
                    nxt = self._move(s, act)
                    done = nxt in (-1, self.N)
                    reward = 1.0 if nxt == self.N else 0.0
                    if nxt in agg:            
                        agg[nxt][0] += prob
                    else:
                        agg[nxt] = [prob, reward, done]
                P[s][a] = [(p, ns, r, d) for ns, (p, r, d) in agg.items()]
        return P          

    def step(self, _a=None):
        self.s += int(self.rng.choice([-1, 1]))
        if self.s == 0:
            return self.s, 0.0, True
        if self.s == self.N + 1:
            return self.s, 1.0, True
        return self.s, 0.0, False

    def true_V(self, gamma):
        """Exact V^pi for states 1..N by linear solve (ground truth for grading)."""
        N = self.N
        P = np.zeros((N, N))                            # non-terminal transitions
        r = np.zeros(N)
        for i, s in enumerate(range(1, N + 1)):
            for ns in (s - 1, s + 1):
                if ns == 0:
                    continue                            # left end: reward 0, absorbing
                if ns == N + 1:
                    r[i] += 0.5 * 1.0                   # right end: reward +1
                    continue
                P[i, ns - 1] += 0.5
        return np.linalg.solve(np.eye(N) - gamma * P, r)

In [9]:
seed = 0
rng = np.random.default_rng(seed)
env = RandomWalk()
V = env.true_V(gamma=1.0)
V

array([0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55,
       0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

# Value Iteration

In [10]:
def q_from_v(
    env: RandomWalk,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    for a in range(env.nA):
        for prob, ns, r, done in env.P[s][a]:
            q[a] += prob * (r + gamma * (V[ns] if not done else 0))
    return q

def value_iteration(
    env: RandomWalk,
    gamma=0.9,
    theta=1e-6,
    max_iters=10000,
):
    V = np.zeros(env.N)
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        for s in range(env.N):
            v_old = V[s]
            V[s] = np.max(q_from_v(env, V, s, gamma))
            delta = max(delta, abs(V[s] - v_old))
        i += 1
    converged = delta < theta
    return V, converged

In [11]:
V_calc, converged = value_iteration(env, gamma=1.0)
np.round(V_calc, 2)

array([0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55,
       0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [12]:
print(
    f"  check the exact solve against the closed form s/20: max diff "
    f"{np.abs(V - np.arange(1, 20) / 20).max():.2e}"
)

  check the exact solve against the closed form s/20: max diff 2.22e-16


# Derivation for s/20


Step 1: at gamma=1, "value" here means "probability of winning"
---------------------------------------------------------------

Every episode ends one of two ways: you reach 20 and collect **+1**, or you reach 0 and collect **0**. Nothing else ever pays. And `gamma=1` means no shrinking, so the return of an episode is literally 1 or 0.

`V(s)` is the _average_ return over many episodes from `s`. Averaging a bunch of 1s and 0s just gives you the fraction that were 1s:

    V(s) = probability you reach 20 before you reach 0, starting from s
    

Quick sanity check: state 10 is dead center, and the walk is 50/50, so it's a coin flip — `V(10)` should be 0.5. Your printout: `0.5`. Good.

Step 2: the one rule the walk gives you
---------------------------------------

Stand on state `s`. You flip a fair coin: half the time you step to `s-1`, half to `s+1`. After that step, your chance of winning is whatever it is _from there_. So:

    chance of winning from s  =  half of (chance from s-1)  +  half of (chance from s+1)
    
    V(s) = ( V(s-1) + V(s+1) ) / 2
    

In words: **every state's value is the average of its two neighbours.** Check it on your numbers at s=10:

    ( V(9) + V(11) ) / 2  =  ( 0.45 + 0.55 ) / 2  =  0.50  =  V(10)   ✓
    

Try it anywhere else and it holds. That's the whole physics of this problem.

Step 3: what does "every point is the average of its neighbours" force?
-----------------------------------------------------------------------

This is the only step with any real content, so let's take it slowly. Multiply that equation by 2:

    2 V(s) = V(s-1) + V(s+1)
    

Now split the left side into `V(s) + V(s)` and move one of each to the other side:

    V(s) - V(s-1)  =  V(s+1) - V(s)
       ^^^                ^^^
       the step you just took     the step you're about to take
    

Read that in plain English: **the gap between `s-1` and `s` is the same as the gap between `s` and `s+1`.** Every gap equals the next gap. So every gap in the whole chain is the same number.

A sequence where every step up is the same size is a straight line. Look at your actual values:

    V:      0.05  0.10  0.15  0.20  0.25  ...  0.90  0.95
    gaps:      0.05  0.05  0.05  0.05      ...     0.05
    

All gaps 0.05. Not approximately — exactly.

Step 4: the two ends fix which straight line
--------------------------------------------

We know two values for free, no math needed:

*   `V(0) = 0` — you've already lost, probability of winning is 0.
*   `V(20) = 1` — you've already won.

So: start at 0, climb to 1, in **20 equal steps** (from 0 to 20). Each step must be `1/20 = 0.05`. After `s` steps you're at `0.05 * s`:

    V(s) = s/20
    

That's it. `V(7) = 7/20 = 0.35`. Check the array: `0.35`. ✓

The 20 is the distance between the two ends (0 and 20), not the 19 interior states — that's the only place people usually slip.